In [ ]:
import argparse
def args_parser():
    parser = argparse.ArgumentParser()
    # federated arguments
    parser.add_argument('--epochs', type=int, default=10, help="rounds of training")
    parser.add_argument('--num_users', type=int, default=100, help="number of users: K")
    parser.add_argument('--frac', type=float, default=0.1, help='the fraction of clients: C')
    parser.add_argument('--local_ep', type=int, default=5, help="the number of local epochs: E")
    parser.add_argument('--local_bs', type=int, default=10, help="local batch size: B")
    parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
    parser.add_argument('--momentum', type=float, default=0.5, help='SGD momentum (default: 0.5)')

    # model arguments
    parser.add_argument('--model', type=str, default='mlp', help='model name')
    parser.add_argument('--kernel_num', type=int, default=9, help='number of each kind of kernel')
    parser.add_argument('--kernel_sizes', type=str, default='3,4,5',
                        help='comma-separated kernel size to use for convolution')
    parser.add_argument('--norm', type=str, default='batch_norm', help="batch_norm, layer_norm, or None")
    parser.add_argument('--num_filters', type=int, default=32,
                        help="number of filters for conv nets -- 32 for miniimagenet, 64 for omiglot.")
    parser.add_argument('--max_pool', type=str, default='True',
                        help="Whether use max pooling rather than strided convolutions")

    # other arguments
    parser.add_argument('--dataset', type=str, default='mnist', help="name of dataset")
    parser.add_argument('--iid', type=int, default=0, help='whether i.i.d or not, 1 for iid, 0 for non-iid')
    parser.add_argument('--num_classes', type=int, default=10, help="number of classes")
    parser.add_argument('--num_channels', type=int, default=3, help="number of channels of images")
    parser.add_argument('--gpu', type=int, default=-1, help="GPU ID")
    parser.add_argument('--stopping_rounds', type=int, default=10, help='rounds of early stopping')
    parser.add_argument('--verbose', type=int, default=1, help='verbose print, 1 for True, 0 for False')
    parser.add_argument('--seed', type=int, default=1, help='random seed (default: 1)')
    parser.add_argument('--ag_scalar', type=float, default=1.0, help="global aggregation updating scalar, simplicity for A Matrix")
    parser.add_argument('--lg_scalar', type=float, default=1.0, help="client local updating scalar, simplicity for S Matrix")
    parser.add_argument('--algorithm', type=str, default='fedavg', help='algorithm for optimization')

    args, _ = parser.parse_known_args()  # Extract only the args
    return args

args = args_parser()
print(f"Number of epochs: {args.epochs}")

In [ ]:
import os

repo_dir = os.path.join(os.getcwd(), 'UnreliableClientsinFL')
os.chdir(repo_dir)
print("Working directory:", os.getcwd())

In [ ]:
import matplotlib
import random
import time
import matplotlib.pyplot as plt
import os
import copy
import numpy as np
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F
import math
from torch import autograd
from tensorboardX import SummaryWriter
from scipy import stats
from scipy.io import savemat

from sampling import mnist_iid, mnist_noniid, cifar_iid, cifar_noniid, server_test
#from options import args_parser
from Update import LocalFSVGRUpdate, LocalUpdate, ServerUpdate, LocalFedProxUpdate
from FedNets import MLP1, CNNMnist, CNNCifar
from averaging import average_weights



def test(net_g, data_loader, args):
    test_loss = 0
    correct = 0

    for idx, (data, target) in enumerate(data_loader):
        if args.gpu != -1:
            data, target = data.cuda(), target.cuda()
        data, target = autograd.Variable(data), autograd.Variable(target)
        log_probs = net_g(data)
        test_loss += F.nll_loss(log_probs, target, size_average=False).data[0]  # sum up batch loss
        y_pred = log_probs.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

    test_loss /= len(data_loader.dataset)
    print('\nTest set: Average loss: {:.4f} \nAccuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
    return correct, test_loss


def calculate_avg_grad(users_g):
    avg_grad = np.zeros(users_g[0][1].shape)
    total_size = np.sum([u[0] for u in users_g])  # Total number of samples of all users
    for i in range(len(users_g)):
        avg_grad = np.add(avg_grad,
                          users_g[i][0] * users_g[i][1])  # += users_g[i][0] * users_g[i][1]   <=> n_k * grad_k
    avg_grad = np.divide(avg_grad, total_size)  # /= total_size
    # print("avg_grad:", avg_grad)
    return avg_grad


def traverseList(nestList):
    flatList = []
    for item in nestList:
        if isinstance(item, list):
            flatList.extend(traverseList(item))
        else:
            flatList.append(item)
    return flatList


def flat(w):
    a = []
    for this_key in w.keys():
        a.append(list(copy.deepcopy(w[this_key].flatten().cpu().numpy())))
    return traverseList(a)


np.set_printoptions(threshold=100000)

if __name__ == '__main__':
    start_time = time.time()
    args = args_parser()
    algo_list = ['fedavg', 'fedprox', 'fsvgr']
    path_project = os.path.abspath('..')
    summary = SummaryWriter('local')
    args.gpu = 0  # -1 for CPU, 0 for first GPU
    args.lr = 0.002  # 0.001 for cifar dataset
    args.model = 'cnn'  # 'mlp' or 'cnn'
    args.dataset = 'mnist'  # 'cifar' or 'mnist'
    args.num_users = 8
    args.local_bs = 5
    args.probability = 1  # Probability of malicious user attack
    args.global_test = 2000
    args.noise = 0.05

    args.malicious = True
    args.defense = False
    # args.malicious_num = [7,16]
    args.iter_num = 0
    args.q = 1
    args.iid = True
    args.verbose = False
    args.num_total_users = 8
    args.num_in_groups = 2
    args.malicious_num = [5, 6]


    print("dataset:", args.dataset, " num_users:", args.num_total_users, "local_bs:", args.local_bs)
    if args.malicious_num:
        print("malicious_num", args.malicious_num[0],args.malicious_num[1] ,"iter_num", args.iter_num, "probability", args.probability)


    def Function(probe_local_ep, final_local_ep, cleanup_rounds, avg_window, T):
        first_time = True
        dict_users = {}
        dataset_train = []
        w_globals = []
        accuracy_list = []
        loss_list = []
        malicious = 0
        malicious_number = -1
        selected = 0
        s_selected = 1
        exclude_value = []
        glob_index = 2
        last_2_group = []
        exit_ = 0
        cıksın_mı_kalsın_mı = 0
        p = 0
        # w_locals snapshots requested for later comparison work:
        # - first_w_locals: the last full-group w_locals computed before the malicious search starts (end of warmup).
        # - latest_w_locals: overwritten every iteration once malicious users have been excluded (phase 3).
        first_w_locals = None
        latest_w_locals = None
        # Caches each user's local update for the current probing baseline, so the group-testing
        # bisection doesn't retrain the same user multiple times against the same starting net_glob.
        phase2_cache = {}
        # Trailing-window (Polyak-style) average of phase-3 w_glob checkpoints, tracked purely for
        # evaluation/reporting -- doesn't change what net_glob trains from next round. Averaging
        # over rounds should shrink the round-to-round noise the same way averaging over clients does,
        # as long as consecutive rounds' noise is close to independent.
        cleanup_w_globals = []
        smoothed_acc_list = []
        smoothed_loss_list = []
        if args.dataset == 'mnist':
            dataset_train = datasets.FashionMNIST('../data/fashionmnist/', train=True, download=True,
                                                  transform=transforms.Compose([
                                                      transforms.ToTensor(),
                                                      transforms.Normalize((0.1307,), (0.3081,))
                                                  ]))
            dataset_test = datasets.FashionMNIST('../data/fashionmnist/', train=False, download=True,
                                                 transform=transforms.Compose([
                                                     transforms.ToTensor(),
                                                     transforms.Normalize((0.1307,), (0.3081,))
                                                 ]))
            # sample users
            if args.iid:
                dict_users = mnist_iid(dataset_train, args.num_total_users)
            else:
                dict_users = mnist_noniid(dataset_train, args.num_total_users)
        else:
            exit('Error: unrecognized dataset')

        img_size = dataset_train[0][0].shape
        net_glob = None

        if args.model == 'cnn' and args.dataset == 'mnist':
          if args.gpu != -1:
              torch.cuda.set_device(args.gpu)
              net_glob = CNNMnist(args=args).cuda()
              # # print(f"Using GPU: {torch.cuda.get_device_name(args.gpu)}")
          else:
              net_glob = CNNMnist(args=args)
        else:
            exit('Error: unrecognized model')
        # print("Nerual Net:", net_glob)
        net_glob.train()  # Train() does not change the weight values
        w_glob = net_glob.state_dict()
        w_size = 0
        for k in w_glob.keys():
            size = w_glob[k].size()
            if (len(size) == 1):
                nelements = size[0]
            else:
                nelements = size[0] * size[1]
            w_size += nelements * 4

        dataset_c_test = server_test(dataset_test, args.global_test)
        net_start = copy.deepcopy(net_glob)

        epoch_acc = []
        epoch_loss = []
        server_acc = np.zeros(args.num_total_users + int(args.num_total_users/args.num_in_groups)*3, dtype='float')
        net_server = copy.deepcopy(net_start)
        net_glob = copy.deepcopy(net_start)
        net_smoothed = copy.deepcopy(net_start)
        dataset_users_test = server_test(dataset_test, 2000)
        global_test = copy.deepcopy(dataset_c_test)

        person = [[] for _ in range(args.num_total_users)]
        w_list = []
        return_loss = 1
        iter = 0

        glob_iter_num = int(math.ceil(T / final_local_ep)) + 0
        # Bu değerin çift sayı olması gerek
        # Hiç bir eleman atılmadan bu değer kadar iterasyon başlangıçta yapılır.

        exit_condition = int(glob_iter_num + np.log2(args.num_total_users) + 1)
        epochs = exit_condition + cleanup_rounds
        bulunan_eleman = 0
        count_for_different_groups = True
        ilk_tur = True
        y11 = []
        y12 = []
        tek_mi_cift_mi = 0
        w_all_locals, loss_all_locals, acc_all_locals = [], [], []
        groups_selected =[[1, args.num_total_users/2], [args.num_total_users/2 + 1, args.num_total_users]]


        # for iter in range(1, epochs + 1): # 0, 1, 2, 3, 4 ...
        while iter <= epochs : # Seçilen epoch sayısına göre loss değerini ayarlamak gerekmekte.
            iter += 1
            epoch_server_acc = []
            w_locals, loss_locals, acc_locals = [], [], []

            # print(f"Loss list = {loss_list}, iter = {iter}")
            print(f"Acc list = {accuracy_list}, iter = {iter}")

            if (iter) % 2 == 0 and (iter > glob_iter_num + 1) and (bulunan_eleman != 2):
              # print("Genel 2 epoch'da 1 kontrol noktası")

              sel_1 = iter - glob_iter_num -3
              # # print(f"{sel_1}, {sel_1 + 1}")
              loss_farki = loss_list[sel_1] - loss_list[sel_1 + 1]
              accuracy_farki = accuracy_list[sel_1] - accuracy_list[sel_1 + 1]

              if tek_mi_cift_mi != 2:
                if abs(accuracy_farki) < 0.1: #abs(loss_farki) < 0.035 :
                  #Hata listeleri arasındaki fark az, demek ki iki grupta birbirine çok benziyor. İki dalda da 1 tane malicious var.
                  tek_mi_cift_mi = 2
                elif abs(accuracy_farki) >= 0.1:#abs(loss_farki) >= 0.035 :
                  #Hata listeleri arasındaki fark fazla, demek ki iki grupta birbirine benzemiyor. Tek dalda iki malicious var..
                  tek_mi_cift_mi = 1
              # print(f"İterasyon = {iter}, maliciousların olduğu grup sayısı => {tek_mi_cift_mi}")
              if tek_mi_cift_mi == 1:
                  malicious = sel_1 + 1 if accuracy_farki > 0 else sel_1 #sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  # print(f"Tek grup için adamlar {malicious_range} grubunda")

              elif tek_mi_cift_mi == 2:

                if ilk_tur == True:
                  malicious_range = [iter - glob_iter_num - 2 - 1, iter - glob_iter_num - 2]
                  # İlk turu fark ettiğimiz iter numarası önemli.Çünkü ona göre malicious gruplar seçilecek.
                  # -glob_iter_num çünkü başta o kadar boş tur dönüyor,
                  # -2 çünkü o ana kadar ki tek grubu 2 iter de parça parça kontrol ediyor,- 1 çünkü parça parça kontolden sonra işleme başlıyor.
                  ilk_tur = False

                elif first_time == True:
                  count_for_different_groups = True
                  malicious = sel_1 + 1 if accuracy_farki > 0 else sel_1 #sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  # print(f"Çift grup için adamlar1 {malicious_range} grubunda")
                  first_time = False

                else:
                  malicious = sel_1 + 1 if accuracy_farki > 0 else sel_1 #sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  # print(f"Çift grup için adamlar2 {malicious_range} grubunda")
                  first_time = True
                  count_for_different_groups = False

              # Fonksiyon haline getir. Tek dal için 2 iter de 1, Çift dal için 4 iter de 1.

              for malicious in malicious_range:

                  idx = malicious
                  group_first_add = int(groups_selected[idx][0])
                  group_second_add = int(groups_selected[idx][0] + groups_selected[idx][1] - 1) / 2
                  group_third_add = group_second_add + 1
                  group_fourth_add = groups_selected[idx][1]

                  # print(f"\nHesaplamalar.\n{group_first_add}--{group_second_add}--{group_third_add}--{group_fourth_add}")

                  if (group_second_add != int(group_second_add)):
                    # print("İlk malicious grubu bulundu baba!\n")
                    groups_selected.append([group_first_add, group_first_add])
                    groups_selected.append([group_first_add + 1, group_first_add + 1])
                    bulunan_eleman += 1
                    if (bulunan_eleman == 1 or bulunan_eleman == 2):
                      exclude_value.append(group_first_add)
                      # print(f"Exculude values = {exclude_value}")
                  elif (group_first_add == group_second_add) and (group_third_add == group_fourth_add) and (tek_mi_cift_mi == 2):
                    # print("İki ayrı dalda iki malicious bulundu")
                    groups_selected.append([group_first_add, group_second_add])
                    groups_selected.append([group_third_add, group_fourth_add])
                  elif (group_first_add == group_second_add) and (tek_mi_cift_mi == 1):
                    # print("Tek dalda iki malicious bulundu")
                    bulunan_eleman = 2
                    exclude_value = groups_selected[malicious]#groups_selected[-1] if malicious == 2 else groups_selected[-2]
                  else:
                    # # print("NET bulunmadı devam")
                    groups_selected.append([group_first_add, group_second_add])
                    groups_selected.append([group_third_add, group_fourth_add])


            # print(f"Gruplar = {groups_selected}")
            if iter > 0 and iter <=glob_iter_num:
              # print("\nGenel Eğitim\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              in_phase2 = False
              current_local_ep = probe_local_ep
            elif ((iter > glob_iter_num) and (bulunan_eleman != 2)):
              # print("\n2 ye bölmeli eğitim\n")
              start_condition1 = int(groups_selected[iter -glob_iter_num -1][0])
              finish_condition1 = int(groups_selected[iter -glob_iter_num -1][1])
              in_phase2 = True
              current_local_ep = probe_local_ep
            else:
              # print(f"\nAtıldıktan sonraki eğitim => {exclude_value}\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              cıksın_mı_kalsın_mı = 1
              in_phase2 = False
              current_local_ep = final_local_ep
            # print(f"Başlangıç değeri = {start_condition1}, Bitiş değeri = {finish_condition1}, local_ep = {current_local_ep}, in_phase2 = {in_phase2}")
            for rounds in range(1, 1 + 1):
                for idx in range(start_condition1, finish_condition1 + 1):
                    if idx in exclude_value:
                      continue
                    client_selected = idx

                    if in_phase2 and client_selected in phase2_cache:
                        # print("Seçilen kullanıcı:", client_selected, "(cache'den)")
                        w, loss, acc = phase2_cache[client_selected]
                        w = copy.deepcopy(w)
                    else:
                        local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[client_selected - 1], tb=summary)
                        w, loss, acc = local.update_weights(net=copy.deepcopy(net_glob), local_ep=int(current_local_ep))
                        if client_selected in args.malicious_num:
                            for this_key in w.keys():
                                dev = np.random.normal(0, args.noise, w[this_key].size()) # 0.2, 0.05 sapıttı,      0.01 düzgün
                                dev = torch.from_numpy(dev).float()
                                if args.gpu != -1:
                                    dev = dev.cuda()
                                w[this_key] = w[this_key] * (-1.5) + dev
                        else:
                          for this_key in w.keys():
                                dev = np.random.normal(0, args.noise, w[this_key].size())
                                dev = torch.from_numpy(dev).float()
                                if args.gpu != -1:
                                    dev = dev.cuda()
                                w[this_key] = w[this_key] + dev
                        if in_phase2:
                            phase2_cache[client_selected] = (copy.deepcopy(w), loss, acc)

                    w_locals.append(copy.deepcopy(w))
                    loss_locals.append(copy.deepcopy(loss))
                    acc_locals.append(copy.deepcopy(acc))
                w_all_locals.append(copy.deepcopy(w_locals))
                loss_all_locals.append(copy.deepcopy(loss_locals))
                acc_all_locals.append(copy.deepcopy(acc_locals))

            if iter == glob_iter_num:
                first_w_locals = copy.deepcopy(w_locals)

            if cıksın_mı_kalsın_mı == 1:
                latest_w_locals = copy.deepcopy(w_locals)

            for idx in range(finish_condition1 - start_condition1 + 1 - len(exclude_value)*cıksın_mı_kalsın_mı):
                net_server.load_state_dict(w_all_locals[iter - 1][idx])
                server = ServerUpdate(args=args, dataset=dataset_test, idxs=dataset_users_test, tb=summary)
                acc, loss = server.test(net=net_server)
                # server_acc[server_acc_num + idx] += acc
                epoch_server_acc.append(acc)
            weight = copy.deepcopy(epoch_server_acc)


            for i in range(len(weight)):
                weight[i] = 1
            total = sum(weight)
            for i in range(len(weight)):
                weight[i] /= total  # same weight average
            for i in range(len(w_locals)):
                w_list.append(copy.deepcopy(flat(w_locals[i])))
            # print(f"weight = {len(weight)}, locals = {len(w_locals)}")
            w_glob = average_weights(w_locals, weight)
            w_globals.append(copy.deepcopy(w_glob))
            net_glob.load_state_dict(w_glob)
            net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            return_acc, return_loss = net_local.test(net=net_glob)
            # print("\nEpoch: {}, Global test loss {}, Global test acc: {:.2f}%\n\n\n\n".format(iter, return_loss, 100 * return_acc))

            if cıksın_mı_kalsın_mı == 1:
                cleanup_w_globals.append(copy.deepcopy(w_glob))
                window = cleanup_w_globals[-avg_window:]
                smoothed_w_glob = average_weights(window, [1.0 / len(window)] * len(window))
                net_smoothed.load_state_dict(smoothed_w_glob)
                smoothed_acc, smoothed_loss = net_local.test(net=net_smoothed)
                smoothed_acc_list.append(smoothed_acc)
                smoothed_loss_list.append(smoothed_loss)
                # print("Smoothed (trailing {} rounds) -- acc: {:.2f}%, loss: {:.4f}\n".format(
                    # len(window), 100 * smoothed_acc, smoothed_loss))

            if iter > glob_iter_num and bulunan_eleman != 2:
              loss_list.append(return_loss)
              accuracy_list.append(return_acc)
              w_glob = w_globals[iter - glob_index]
              glob_index += 1
              net_glob.load_state_dict(w_glob)
              net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)

            y11.insert(iter, return_loss)
            y12.insert(iter, return_acc)

            if iter <= 200:
                if iter % 1 == 0 or iter == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
            else:
                if iter % 1 == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
        a = epoch_acc[-1]
        b = epoch_loss[-1]
        return a, b, y11, y12, epoch_acc, epoch_loss, first_w_locals, latest_w_locals, smoothed_acc_list, smoothed_loss_list, exclude_value
    probe_local_ep = 20     # small local_ep for warmup + group-testing probes (speed)
    final_local_ep = 8      # local_ep for the post-detection cleanup phase (was 256): that many
                            # local epochs per round on a tiny per-client shard was overfitting
                            # hard before every aggregation, which was driving the accuracy wobble
    cleanup_rounds = 1    # far more (much cheaper) aggregation rounds instead, so cleanup
                            # converges on many small updates rather than a few huge ones
    avg_window = 10         # trailing window (rounds) for Polyak-style smoothing of the phase-3 model
    true_list = []
    for i in range(60):
        (acc, loss, y1, y2, a, b, first_w_locals, latest_w_locals,
         smoothed_acc_list, smoothed_loss_list, excluded_value) = Function(probe_local_ep, final_local_ep, cleanup_rounds, avg_window, final_local_ep * 3)
        end_time = time.time()

        if excluded_value == args.malicious_num:
            print(f"{i+1}) dogru buldu.\t {excluded_value} {args.malicious_num}")
            true_list.append(1)
        else:
            true_list.append(0)
            print(f"{i+1}) yanlis buldu.\t {excluded_value} {args.malicious_num}")
        elapsed_time = end_time - start_time
        minutes = int(elapsed_time // 60)
        seconds = int(elapsed_time % 60)
        print(f"{minutes} dk, {seconds} sn")
    print(f"%{(sum(true_list)/len(true_list))*100}")
    savemat(repo_dir + rf"\true_labels\{args.noise}_2_malicious_rate.mat", {"data": np.array(true_list)})

    if smoothed_acc_list:
        raw_phase3_acc = y2[-len(smoothed_acc_list):]
        plt.figure()
        plt.plot(raw_phase3_acc, label='raw (per-round)')
        plt.plot(smoothed_acc_list, label=f'trailing-{avg_window}-round average')
        plt.xlabel('cleanup-phase round')
        plt.ylabel('global test accuracy')
        plt.legend()
        plt.title('Raw vs. smoothed accuracy after malicious users excluded')
        plt.show()


In [ ]:
# torch.save(first_w_locals, "first_w_locals.pt")
# torch.save(latest_w_locals, "latest_w_locals.pt")

In [ ]:
# np.save("acc_history_detect_exclude.npy", np.array(a))
# print(f"Saved {len(a)} rounds of accuracy history to acc_history_detect_exclude.npy")

In [ ]:
# first_w_locals = torch.load("first_w_locals.pt", map_location="cpu")
# latest_w_locals = torch.load("latest_w_locals.pt", map_location="cpu")